# Kenar dibi takibi — ACT eğitimi (Colab / Kaggle)

`docs/learning.md`, 6. aşama, dilim **6c**. Bu defter robottan ya da simülatörden gelen bölümleri
LeRobot veri setine çevirir, **ACT** politikasını eğitir ve ayrılmış bölümlerde **çevrimdışı** puanlar.

**Gerekenler**
- GPU'lu çalışma ortamı (Colab: *Runtime → Change runtime type → GPU*; Kaggle: *Accelerator → GPU*).
- Python **≥ 3.12** (LeRobot 0.6.1'in şartı — aşağıdaki ilk hücre kontrol eder).
- `dataset.zip`: dev container'da üretilir:
  ```bash
  ros2 run robot_sim edge_batch --lawns 20 --out ~/.robot/sim_episodes      # sim'de uzman bölümleri (6b)
  ros2 run robot_learning episode_export ~/.robot/sim_episodes -o ~/dataset   # npz + mp4 + meta.json (5d)
  cd ~ && python3 -m zipfile -c dataset.zip dataset/                         # (container'da `zip` yok)
  ```
  sonra Google Drive'a ya da Kaggle'a "Dataset" olarak yükle (Kaggle zip'i kendisi açar; defter
  ikisini de tanır).

⚠️ Buradaki puan **açık çevrim**dir: politika uzmanın kendi durumlarında uzmanla ne kadar aynı şeyi söylüyor.
Hata birikmesini göstermez — o, simülatörde kapalı çevrimde (6d) ölçülür.

In [ ]:
import shutil, subprocess, sys
print(sys.version)
assert sys.version_info >= (3, 12), "LeRobot 0.6.1 needs Python >= 3.12 — bu ortam uygun değil"
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    print("⚠️ GPU YOK — çalışma ortamını GPU'ya çevir; CPU'da eğitim saatler sürer")

In [ ]:
# Sürüm sabit: LeRobot'un veri seti formatı sürümler arasında değişti.
# torch'u LeRobot'un istediği aralığa çekebilir; Colab "restart" isterse yeniden başlatıp devam et.
!pip install -q "lerobot[dataset,training]==0.6.1"

In [ ]:
import os, pathlib
# --- AYARLAR -----------------------------------------------------------------
REPO_REF = "main"   # betiklerin alınacağı dal / commit (github.com/yusuftiryaki/maraba)
ON_KAGGLE = os.path.exists("/kaggle")
WORK = pathlib.Path(os.environ.get("HOVERBOT_WORK") or ("/kaggle/working" if ON_KAGGLE else "/content"))
# dataset.zip nerede? Kaggle: /kaggle/input/<veri-seti-adı>/dataset.zip
# Colab + Drive: önce aşağıdaki satırları aç.
# from google.colab import drive; drive.mount("/content/drive")
# Kaggle AÇAR yüklenen zip'leri: orada /kaggle/input altında meta.json'u ara.
DATA_ZIP = pathlib.Path(os.environ.get("HOVERBOT_DATA_ZIP") or (
    next((str(p.parent) for p in pathlib.Path("/kaggle/input").rglob("meta.json")), "")
    if ON_KAGGLE else "/content/drive/MyDrive/hoverbot/dataset.zip"))
STEPS = int(os.environ.get("HOVERBOT_STEPS", 20000))  # ACT için makul başlangıç; kayıp eğrisine bak
BATCH = 16
HOLD_OUT = 0.15      # bölümlerin bu kadarı eğitime girmez, puanlamaya ayrılır
# -----------------------------------------------------------------------------
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 2 if DEVICE == "cuda" else 0
print(WORK, DATA_ZIP, DATA_ZIP.exists(), DEVICE)

In [ ]:
import shutil, urllib.request, zipfile
EXPORT = WORK / "export"; LR_ROOT = WORK / "hoverbot_edge"
shutil.rmtree(EXPORT, ignore_errors=True); shutil.rmtree(LR_ROOT, ignore_errors=True)
if DATA_ZIP.is_dir():              # Kaggle: already unpacked
    shutil.copytree(DATA_ZIP, EXPORT)
else:
    zipfile.ZipFile(DATA_ZIP).extractall(EXPORT)
for name in ("to_lerobot.py", "eval_offline.py"):
    local = os.environ.get("HOVERBOT_SCRIPTS")          # repo'daki learning/ ya da veri setindeki kopya
    if local:
        shutil.copy(pathlib.Path(local) / name, WORK / name)
    else:
        url = f"https://raw.githubusercontent.com/yusuftiryaki/maraba/{REPO_REF}/learning/{name}"
        urllib.request.urlretrieve(url, WORK / name)
# Zip'in üst dizini ne olursa olsun: meta.json'un olduğu yer veri setidir.
EXPORT = next(p.parent for p in EXPORT.rglob("meta.json"))
print(EXPORT, sorted(p.name for p in EXPORT.iterdir()))

In [ ]:
# npz + mp4 -> LeRobot veri seti. Gözlem: kamera + yalnız ivmeölçer (varsayılan). Konum yok: politika
# tek bir sahanın koordinatlarını ezberlemesin. Hız da yok: ilk politika kendi hızını kopyalamayı öğrendi
# (learning.md 6d, "copycat"). Etiket: sim'de uzman.
!cd {WORK} && python to_lerobot.py {EXPORT} --out {LR_ROOT} --repo-id local/hoverbot_edge

In [ ]:
import json
summary = json.load(open(LR_ROOT / "hoverbot_conversion.json"))
n = len(summary["episodes"])
held = max(1, round(n * HOLD_OUT)) if n > 1 else 0
TRAIN_EPISODES = list(range(n - held))
EVAL_EPISODES = list(range(n - held, n)) or [0]
print(f"{n} bölüm: eğitim {TRAIN_EPISODES}, puanlama {EVAL_EPISODES}")
EPISODES_ARG = "[" + ",".join(map(str, TRAIN_EPISODES)) + "]"

## ACT
`chunk_size=20` (10 Hz'de 2 s ufuk), `n_action_steps=5` (her 0.5 s'de yeniden tahmin).
⚠️ ACT'nin varsayılanı 100 / 100: 10 Hz'de **10 saniye kör sürüş** — bu robot için kabul edilemez.

In [ ]:
OUT = WORK / "outputs" / "act_edge"
shutil.rmtree(OUT, ignore_errors=True)
!lerobot-train \
  --dataset.repo_id=local/hoverbot_edge --dataset.root={LR_ROOT} "--dataset.episodes={EPISODES_ARG}" \
  --policy.type=act --policy.chunk_size=20 --policy.n_action_steps=5 \
  --policy.device={DEVICE} --policy.push_to_hub=false \
  --steps={STEPS} --batch_size={BATCH} --num_workers={WORKERS} \
  --log_freq=200 --save_freq=5000 \
  --output_dir={OUT} --job_name=act_edge --wandb.enable=false

In [ ]:
# Çevrimdışı puan: ayrılmış bölümlerde politikanın aksiyonu ile uzmanınki (MAE) ve çıkarım süresi.
EVAL_ARG = ",".join(map(str, EVAL_EPISODES))
!cd {WORK} && python eval_offline.py {OUT}/checkpoints/last/pretrained_model \
    --dataset {LR_ROOT} --episodes {EVAL_ARG} --device {DEVICE}

## (İsteğe bağlı) Diffusion Policy
Sol/sağ ikilemlerinde (ağacın hangi yanından) ACT'nin ortalama alması sorun olursa.
`crop_shape` mutlaka ver: varsayılan 84×84, 320×240 karemizden küçük bir parça keser.

In [ ]:
RUN_DIFFUSION = False
if RUN_DIFFUSION:
    OUT_D = WORK / "outputs" / "diffusion_edge"
    shutil.rmtree(OUT_D, ignore_errors=True)
    !lerobot-train \
      --dataset.repo_id=local/hoverbot_edge --dataset.root={LR_ROOT} "--dataset.episodes={EPISODES_ARG}" \
      --policy.type=diffusion --policy.crop_shape="[224, 288]" --policy.horizon=16 --policy.n_action_steps=8 \
      --policy.device={DEVICE} --policy.push_to_hub=false \
      --steps={STEPS} --batch_size={BATCH} --num_workers={WORKERS} --log_freq=200 --save_freq=5000 \
      --output_dir={OUT_D} --job_name=diffusion_edge --wandb.enable=false

In [ ]:
# Robota (6d: policy_node) götürülecek paket: politika + dönüşüm özeti.
PACK = WORK / "act_edge_policy.zip"
# ⚠️ Özet ağırlıkların YANINA: policy_node durum sütunlarını buradan okur (yanlış tahmin sessizdir).
shutil.copy(LR_ROOT / "hoverbot_conversion.json", OUT / "checkpoints" / "last" / "pretrained_model")
shutil.make_archive(str(PACK.with_suffix("")), "zip", OUT / "checkpoints" / "last" / "pretrained_model")
shutil.copy(LR_ROOT / "hoverbot_conversion.json", WORK / "act_edge_conversion.json")
print(PACK, PACK.stat().st_size // 1_000_000, "MB")
# Colab: from google.colab import files; files.download(str(PACK))
# Kaggle: /kaggle/working altındakiler "Output" sekmesinden indirilir.